# 2D data and magnetic SANS

IQuana fits **2D detector images** I(qx, qy) and **polarised (magnetic) SANS**.
The two arrive together because sasmodels evaluates magnetic scattering only in
its 2D kernel: with a 1D curve the magnetic parameters have no effect at all.

This notebook walks through:

1. Loading a measured 2D image, masking it, and fitting it
2. What the magnetic parameters mean, shown on simulated detector patterns
3. A magnetic fit that recovers known truth
4. The checks that stop a magnetic fit from silently meaning something else
5. Fitting measured SANSPOL data of magnetic nanoparticles
6. Saving the analysis, mask and magnetism included

In [ ]:
import os
import tempfile
import warnings

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from iquana import IQuana, examples, set_verbosity

# Keep the per-call progress messages out of the way; warnings still show.
set_verbosity('quiet')

## 1. A measured 2D image

The bundled `p123_2d` example is a 100 × 100 detector image of Pluronic P123
micelles in D2O. `load_data()` recognises 2D files through sasdata, so nothing
changes in how you load them.

In [ ]:
fitter = IQuana()
fitter.load_data(examples.example_path('p123_2d'))
print('2D data:', fitter.is_2d)
print('|Q| range:', fitter.get_q_range())
{key: value for key, value in fitter.get_mask().items() if key != 'excluded'}

### Selecting pixels

A detector image nearly always needs a mask: the beam stop, and here 33 pixels
whose intensity and uncertainty are both zero. The bumps engine cannot weight a
zero-σ pixel, and IQuana never masks anything silently, so the choice is
stated:

- `mask_ring(qmax=...)` excludes a disc around the beam
- `mask_invalid()` excludes pixels whose dI is zero or not finite
- `mask_box(...)` and `set_mask(...)` cover the rest; `reset_mask()` undoes them

The selection belongs to the fitter, not the dataset: the file's own mask is
never rewritten, and the selection is saved with the analysis.

In [ ]:
fitter.mask_ring(qmax=0.006)
fitter.mask_invalid()
mask = fitter.get_mask()
print(f"{mask['fitted']} of {mask['pixels']} pixels fitted, {mask['user']} excluded by masks")
fitter.plot_results()   # no model yet: the data alone, masked pixels transparent

### Model, preview and fit

The rest is the 1D workflow. `plot_model()` draws three panels, data, model
and residuals, on a shared log colour scale with equally scaled q axes. The
file has no resolution columns, so we state that it is fitted unsmeared.

In [ ]:
fitter.set_model('sphere')
fitter.set_structure_factor('hardsphere')
fitter.set_resolution('none')
fitter.set_param('sld', value=0.4)
fitter.set_param('sld_solvent', value=6.4)
fitter.set_param('scale', value=0.05, min=1e-6, max=10, vary=True)
fitter.set_param('background', value=0.5, min=0, max=10, vary=True)
fitter.set_param('radius', value=50, min=10, max=200, vary=True)
fitter.set_param('radius_effective', value=60, min=10, max=300, vary=True)
fitter.set_param('volfraction', value=0.2, min=0, max=0.6, vary=True)
fitter.plot_model()

In [ ]:
result = fitter.fit()
fitter.plot_results()

In [ ]:
fitter.get_fit_report()

`set_q_range()` now selects an **annulus** in |Q|, and `calculate()`,
`save_results()` and `report()` work as for 1D data. The CSV has `Qx` and
`Qy` columns and one row per fitted pixel.

In [ ]:
fitter.set_q_range(qmin=0.02, qmax=0.1)
print('pixels in the annulus:', fitter.get_mask()['fitted'])
fitter.reset_q_range()

## 2. What the magnetic parameters mean

sasmodels gives every compiled model with an SLD a set of magnetic
parameters. IQuana keeps them out of the way until you ask for them:

In [ ]:
demo = IQuana()
demo.set_data(examples.simulate_2d('sphere', npixels=8))   # any 2D data
demo.set_model('sphere')
print('supports magnetism:', demo.supports_magnetism())
print('in the model now:  ', 'sld_M0' in demo.params)
print('available:         ', demo.get_magnetic_parameters())

| Parameter | Meaning |
|---|---|
| `up_frac_i`, `up_frac_f` | fraction of spin-up neutrons before and after the sample |
| `up_theta`, `up_phi` | direction of the polarisation axis **P**, degrees |
| `<sld>_M0` | magnetic SLD of that region, 10⁻⁶ Å⁻² |
| `<sld>_mtheta`, `<sld>_mphi` | direction of that region's magnetisation **M**, degrees |

**Convention**: z is the beam, x and y the detector axes; a direction
(θ, φ) is (sin θ cos φ, sin θ sin φ, cos θ), for **M** and **P** alike.
The defaults put **P** along +x and **M** along the beam.

Rather than remember which fractions describe which experiment,
`enable_magnetism(beam=...)` takes a preset: `'unpolarised'`, `'sanspol+'`,
`'sanspol-'` (polariser, no analyser), and `'pa++'`, `'pa--'`, `'pa+-'`,
`'pa-+'` (full polarisation analysis), each scaled by an efficiency.

Below, one magnetised sphere with **M** and **P** along a horizontal field (+x)
is seen by four experiments. The nuclear scattering is isotropic; the
magnetic terms are not, and how they combine with it depends on the spin
channel.

In [ ]:
axis = np.linspace(-0.08, 0.08, 81)
qx, qy = np.meshgrid(axis, axis)
beams = ['unpolarised', 'sanspol+', 'sanspol-', 'pa+-']

figure = make_subplots(rows=1, cols=4, subplot_titles=beams, horizontal_spacing=0.03)
for column, beam in enumerate(beams, start=1):
    demo.enable_magnetism(False)
    demo.enable_magnetism(beam=beam)
    for name, value in {'radius': 60, 'sld': 2, 'sld_solvent': 6, 'background': 1e-3,
                        'sld_M0': 3, 'sld_mtheta': 90, 'sld_mphi': 0}.items():
        demo.set_param(name, value=value)
    image = demo.calculate(qx=qx, qy=qy)
    figure.add_trace(go.Heatmap(x=axis, y=axis, z=np.log10(image), coloraxis='coloraxis'),
                     row=1, col=column)
    figure.update_yaxes(scaleanchor=f'x{column if column > 1 else ""}', row=1, col=column)
figure.update_layout(height=330, width=1100, coloraxis={'colorbar': {'title': 'log₁₀ I'}},
                     title='One magnetised sphere, four spin channels (field along +x)')
figure

SANSPOL+ and SANSPOL− are mirror images of each other: their difference is the
nuclear–magnetic cross term, and it is strongest perpendicular to the field.
The spin-flip channel `pa+-` sees no nuclear scattering at all, only
magnetisation perpendicular to **P**.

## 3. A magnetic fit with known truth

`examples.simulate_2d()` accepts magnetic parameters and a beam preset, and
records the full parameter set in `data.truth`.

In [ ]:
TRUTH = {'radius': 60.0, 'scale': 0.05, 'background': 0.01, 'sld': 2.0, 'sld_solvent': 6.0,
         'sld_M0': 1.5, 'sld_mtheta': 90.0}
image = examples.simulate_2d('sphere', qmax=0.15, npixels=64, beam='sanspol+', **TRUTH)

magnetic = IQuana()
magnetic.set_data(image)
magnetic.set_model('sphere')
magnetic.set_resolution('none')
magnetic.enable_magnetism(beam='sanspol+')
for name in ('sld', 'sld_solvent', 'background', 'sld_mtheta'):
    magnetic.set_param(name, value=TRUTH[name])
magnetic.set_param('radius', value=45, min=10, max=200, vary=True)
magnetic.set_param('scale', value=0.03, min=1e-4, max=1, vary=True)
magnetic.set_param('sld_M0', value=0.5, min=0.05, max=5, vary=True)
magnetic.get_params()

In [ ]:
result = magnetic.fit()
{name: round(float(info['value']), 4) for name, info in result['parameters'].items() if not info['fixed']}

In [ ]:
magnetic.plot_results()

## 4. The checks that keep a magnetic fit honest

Three behaviours of the sasmodels kernel would otherwise change what a fit
means without saying so.

**The kernel switch at zero magnetisation.** With every `*_M0` at zero,
sasmodels runs its non-magnetic kernel, which ignores the polarisation. In
the spin-flip channel the nuclear intensity is then suddenly back. A fit that
lets the magnetisation reach zero there is refused:

In [ ]:
spin_flip = IQuana()
spin_flip.set_data(image)
spin_flip.set_model('sphere')
spin_flip.enable_magnetism(beam='pa+-')
spin_flip.set_param('sld_mtheta', value=90)
spin_flip.set_param('sld_M0', value=1.0, min=0.0, max=5.0, vary=True)
try:
    spin_flip.fit()
except ValueError as error:
    print(error)

**Parameters the model does not respond to.** With **M** along the beam
(`sld_mtheta = 0`) its azimuth changes nothing, so a fit cannot determine it.
IQuana checks every varied magnetic parameter before fitting:

In [ ]:
insensitive = IQuana()
insensitive.set_data(image)
insensitive.set_model('sphere')
insensitive.enable_magnetism(beam='sanspol+')
insensitive.set_param('sld_M0', value=1.5)
insensitive.set_param('sld_mtheta', value=0)
insensitive.set_param('sld_mphi', min=-180, max=180, vary=True)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    insensitive.fit()
print(*[str(warning.message) for warning in caught if 'sld_mphi' in str(warning.message)], sep='\n')

**1D curves are not magnetic.** Asking for a 1D curve of a magnetic model
would return sasmodels' orientation-averaged nuclear curve, so it is refused:

In [ ]:
try:
    magnetic.calculate(q=np.linspace(0.01, 0.1, 10))
except ValueError as error:
    print(error)

## 5. Measured SANSPOL data

The `sanspol_plus` and `sanspol_minus` examples are the polarised images of
magnetic nanoparticles from SasView's magnetism tutorial, measured with a
spin-up and a spin-down incident beam in a horizontal field. Their presets
mask the beam stop, enable magnetism with the right beam, and start from
coarse values. The model and values are a starting point, not a confirmed
result.

In [ ]:
examples.describe('sanspol_plus')

In [ ]:
channels = {}
for name in ('sanspol_plus', 'sanspol_minus'):
    channel = examples.load_fitter(name)
    channel.set_resolution('none')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        channel.fit()
    channels[name] = channel

{name: {key: round(float(channel.params[key]['value']), 4)
        for key in ('radius', 'scale', 'background', 'sld_M0')}
 for name, channel in channels.items()}

In [ ]:
channels['sanspol_plus'].plot_results()

The two channels agree on the particle radius, and the binned display handles
the irregular pixel layout of these files. They disagree on the magnetic SLD
because each was fitted alone: the proper analysis fits both channels
together, sharing the sample parameters, which `MultiFitter` does not yet
support for 2D data.

## 6. Save and reload

`save_analysis()` stores the pixel mask, the magnetism block and the beam
preset. A simulated image lives in memory only, so it is supplied again on
reload, exactly as for 1D data.

In [ ]:
path = os.path.join(tempfile.mkdtemp(), 'magnetic.json')
magnetic.mask_ring(qmax=0.01)
magnetic.fit()
magnetic.save_analysis(path)

restored = IQuana.load_analysis(path, data=image)
print('magnetism enabled:', restored.is_magnetism_enabled())
print('pixels masked:    ', restored.get_mask()['user'])
print('result restored:  ', restored.fit_result is not None)

## Summary

- 2D data uses the 1D workflow; masks are fitter state and saved with the analysis.
- `enable_magnetism(beam=...)` exposes the magnetic parameters for 2D data.
- IQuana refuses or flags magnetic setups sasmodels would evaluate as
  something other than what you asked for.
- Not yet: simultaneous fits of several spin channels, vector-SLD models, and
  the beta approximation in 2D.

See the *2D Data* and *Magnetic (Polarised) SANS* sections of the user guide
for the details.